# GRT Server 빠른 실사용 점검 (임시)

[Colab에서 열기](https://colab.research.google.com/github/YeoJune/GRT/blob/main/notebooks/tmp_server_acceptance.ipynb) · [실행 가이드](https://github.com/YeoJune/GRT_server/blob/main/docs/20261023_aws_test_execution_guide.md)

**GPU 런타임 + `GRT_SERVER_TOKEN` write Secret**을 준비한 뒤 위에서부터 실행합니다. Kaggle은 Internet과 해당 Secret의 notebook 접근도 켭니다. GPU는 1개만 사용합니다.

기본은 다운로드 없는 **합성 token + 실제 작은 GRT 학습**입니다. 정상 업로드, 원본 비교, 작은 posthoc 실패/재시도, 큐 우선순위/pause/drain, clone/retry, 실제 실행 취소, read-only/CSRF/토큰 폐기, 잘못된 파일 검증을 점검합니다. 연구 성능 검사가 아닙니다. 실제 WikiText 입력은 아래 `data_mode='wikitext_sample'`로 새 시험을 시작합니다.

학습·전송 시간은 환경에 따라 달라집니다. 처음 설치 이후 작은 실험들로 구성하며, 취소 전달에는 기본 최대 약 30초와 저장 시간이 필요합니다. 서버 종료·전체 세션 폐기·10분 통신 장애는 실행 가이드에서 따로 진행합니다.

실패하면 그 셀에서 멈추고 오류와 마지막 요약을 전달하세요. 마지막 **결과 요약** 셀은 중간 실패 후에도 실행할 수 있습니다. 노트북 전체나 `.server`/`.grt-server` 디렉터리는 공유하지 마세요.


In [ ]:
from pathlib import Path
from datetime import datetime, timedelta, timezone
from uuid import uuid4
import os, subprocess, sys, time, json, hashlib, copy

server_url = 'https://grt.grtserver.cloud'
runtime = 'kaggle' if Path('/kaggle/working').exists() else 'colab' if Path('/content').exists() else 'cpu'
secret_source = runtime if runtime in ('colab', 'kaggle') else 'env'
device = 'cuda' if runtime in ('colab', 'kaggle') else 'cpu'
data_mode = 'synthetic'  # synthetic | wikitext_sample
run_posthoc = True
run_cancel = True
session_minutes = 60  # 실제 남은 세션 시간 이내로 지정

# 이 셀을 재실행하면 새 시험입니다. 실행 중에는 재실행하지 마세요.
base = Path('/kaggle/working') if runtime == 'kaggle' else Path('/content') if runtime == 'colab' else Path.cwd() / 'runs'
repo = base / 'GRT' if runtime != 'cpu' else Path.cwd()
test_id = 'acceptance-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S') + '-' + uuid4().hex[:6]
work = base / test_id
work.mkdir(parents=True, exist_ok=False)
queue_name = test_id
deadline = (datetime.now(timezone.utc) + timedelta(minutes=session_minutes)).isoformat()
checks, tracked_jobs = {}, []
print('시험:', test_id, '|', runtime, device, data_mode, '| deadline:', deadline)


## 1. 코드·의존성 준비

기존 checkout에 수정이 있으면 멈춥니다. 새 런타임에서 실행하는 것이 가장 간단합니다. 설치 전후에 kernel에서 Transformers를 미리 import하지 마세요. CPU 로컬 실행은 GRT checkout에서 환경변수 Secret을 주입한 뒤 실행합니다.


In [ ]:
if not (repo / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', 'main', '--single-branch',
                    'https://github.com/YeoJune/GRT.git', str(repo)], check=True)
elif runtime != 'cpu':
    dirty = subprocess.check_output(['git', '-C', str(repo), 'status', '--porcelain'], text=True)
    if dirty.strip():
        raise RuntimeError('checkout에 수정이 있습니다. 보존 후 새 경로/런타임에서 실행하세요.')
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only', 'origin', 'main'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(repo) + '[wikitext]'], check=True)
sys.path.insert(0, str(repo / 'src'))
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'


In [ ]:
import requests
import torch
from urllib.parse import quote
from grt.utils.kaggle import WikiTextRun, file_hash
from grt.utils.server import ServerClient, ServerError

if device == 'cuda' and not torch.cuda.is_available():
    raise RuntimeError('GPU 런타임을 선택하세요. CPU는 device를 명시적으로 바꿉니다.')
client = ServerClient(server_url, secret_source=secret_source)
me = client.check()  # Secret 누락/read-only는 학습 전에 여기서 실패
client.ensure_schema()
client.ensure_queue(queue_name)
queue = next(q for q in client.request('GET', '/queues')['data'] if q['name'] == queue_name)
caps = client.request('GET', '/capabilities')
commit = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
print('연결:', server_url, '| 권한:', me['kind'], '| 만료:', me['expires_at'])
print('GRT commit:', commit, '| GPU:', torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU')

def record(name, **details):
    checks[name] = dict(status='PASS', utc=datetime.now(timezone.utc).isoformat(), **details)
    print('PASS:', name, details)

def detail(job_id):
    return client.request('GET', '/jobs/' + job_id)

def wait_for(getter, predicate, timeout=120):
    end = time.monotonic() + timeout
    while True:
        value = getter()
        if predicate(value):
            return value
        if time.monotonic() >= end:
            raise TimeoutError('상태 변경 대기 시간 초과. 서버 상세/로컬 파일을 확인하세요.')
        time.sleep(1)

def track(item):
    if item['id'] not in tracked_jobs:
        tracked_jobs.append(item['id'])
    return item

def transfer(method, url, **kwargs):
    # 서명 URL/예외 원문을 노트북 출력에 남기지 않습니다. Bearer는 보내지 않습니다.
    try:
        response = requests.request(method, url, timeout=(10, 120), allow_redirects=False, **kwargs)
    except requests.RequestException:
        raise RuntimeError('파일 전송 실패: 서명 URL을 출력하지 말고 해당 단계를 재시도하세요.') from None
    if not response.ok or response.is_redirect:
        raise RuntimeError('파일 전송 HTTP ' + str(response.status_code))
    return response

record('connection', runners=caps['runners'])


## 2. 작은 입력 준비

`synthetic`은 128개 token의 반복 입력이고 외부 다운로드가 없습니다. `wikitext_sample`은 Hugging Face WikiText-103의 앞부분을 streaming으로 읽고 GPT-2 tokenizer로 처리합니다. 모델 weights는 다운로드하지 않습니다. 두 경우 모두 원 연구 데이터와 분리된 prepared 경로·manifest를 사용합니다.


In [ ]:
assert data_mode in ('synthetic', 'wikitext_sample')
prepared = work / 'prepared'
prepared.mkdir(exist_ok=True)
tokenization = 'word' if data_mode == 'synthetic' else 'gpt2_bpe'
vocab_size = 128 if data_mode == 'synthetic' else 50257
segment_len = 4 if data_mode == 'synthetic' else 32
counts = dict(train=1024, validation=256, test=128) if data_mode == 'synthetic' else dict(train=4096, validation=2048, test=512)
manifest_path = prepared / 'manifest.json'
if not manifest_path.exists():
    manifest = dict(vocab_size=vocab_size, tokenization=tokenization, token_counts={}, sha256={},
                    test_input=data_mode, note='Acceptance input only; not a research-quality run')
    if data_mode == 'wikitext_sample':
        from datasets import load_dataset
        from transformers import AutoTokenizer
        tokenizer = AutoTokenizer.from_pretrained('openai-community/gpt2')
    for split, count in counts.items():
        if data_mode == 'synthetic':
            tokens = torch.arange(count, dtype=torch.long) % vocab_size
        else:
            source = load_dataset('Salesforce/wikitext', 'wikitext-103-v1', split=split, streaming=True)
            values = []
            for row in source:
                values.extend(tokenizer.encode(row['text'] + '\n', add_special_tokens=False))
                if len(values) >= count:
                    break
            if len(values) < count:
                raise RuntimeError('WikiText sample이 부족합니다.')
            tokens = torch.tensor(values[:count], dtype=torch.long)
        target = prepared / (split + '.pt')
        torch.save(tokens, target)
        manifest['token_counts'][split] = len(tokens)
        manifest['sha256'][split] = file_hash(target)
    manifest_path.write_text(json.dumps(manifest, indent=2))
else:
    manifest = json.loads(manifest_path.read_text())
for split, digest in manifest['sha256'].items():
    assert file_hash(prepared / (split + '.pt')) == digest

# 모델/배치 축소는 이 시험에만 적용합니다. 기존 연구 카탈로그는 바꾸지 않습니다.
def make_run(label, *, steps=2, mode='upload', model='grt'):
    return WikiTextRun(repo, runtime=runtime, device=device, world_size=1,
        work_dir=work / 'local', run_id=test_id + '-' + label,
        size='tiny', budget='smoke', models=(model,), tokenization=tokenization,
        steps=steps, benchmark_steps=1, wandb=False, test_at_end=False,
        server_mode=mode, server_url=server_url, secret_source=secret_source,
        server_queue=queue_name, server_deadline=deadline,
        overrides={'model': {'vocab_size': vocab_size, 'd_model': 32, 'segment_len': segment_len,
                    'num_registers': 0 if model == 'neox' else 3, 'alu': {'num_layers': 1, 'nhead': 4, 'd_ff': 64},
                    **({'router': {'mlp_hidden': 32}} if model == 'grt' else {})},
            'training': {'auto_batch': False, 'batch_size': 1, 'grad_accum_steps': 1,
                         'global_batch_size': 1, 'bptt_segments': 0 if model == 'neox' else 1},
            'evaluation': {'batch_size': 1, 'every_steps': 2}, 'rtla': {'enabled': False}})

record('prepared', input=data_mode, manifest_sha256=file_hash(manifest_path))


## 3. 정상 upload (R02~R04)

아래 셀을 실행한 뒤 서버의 **실험** 화면에서 이번 시험 이름을 검색합니다. 학습과 업로드는 별도 과정이며, 셀이 끝났을 때 서버 상태가 `succeeded`여야 합니다.


In [ ]:
upload_run = make_run('upload')
upload_run.setup(install=False)
upload_run.prepare_data(prepared_input=prepared)
upload_run.preflight()
upload_run.train()
upload_run.report(plot=False)
upload_items = client.request('GET', '/jobs?campaign=' + quote(upload_run.run_dir.name))['data']
assert len(upload_items) == 1
upload_job = track(detail(upload_items[0]['id']))
assert upload_job['status'] == 'succeeded'
assert len(upload_job['attempts']) == 1
record('upload', job_id=upload_job['id'], status_observed=upload_job['status'])


## 4. 원본 다운로드·ZIP·report 재호출 (R05~R06, B07~B08)

모든 verified 파일을 내려받아 서버 hash와 비교합니다. 원본 report·checkpoint는 실행기 파일과도 직접 비교합니다. 서명 URL은 출력하지 않습니다.


In [ ]:
collected = work / 'downloads'
collected.mkdir(exist_ok=True)
verified = [a for a in upload_job['artifacts'] if a['status'] == 'verified']
assert {'report', 'config', 'provenance', 'checkpoint', 'log', 'result'} <= {a['kind'] for a in verified}
for artifact in verified:
    link = client.request('GET', '/artifacts/' + artifact['id'] + '/download')
    payload = transfer('GET', link['url']).content
    assert len(payload) == artifact['size_bytes']
    assert hashlib.sha256(payload).hexdigest() == artifact['sha256']
    (collected / artifact['id']).write_bytes(payload)
    local = upload_run.report_path if artifact['name'] == 'experiment_report.json' else upload_run.run_dir / 'grt' / artifact['name']
    if local.is_file():
        assert file_hash(local) == artifact['sha256']

before = {a['id'] for a in upload_job['artifacts']}
checkpoint_before = file_hash(upload_run.run_dir / 'grt' / 'last.pt')
upload_run.report(plot=False)
repeated = detail(upload_job['id'])
assert {a['id'] for a in repeated['artifacts']} == before
assert len(repeated['attempts']) == 1
assert file_hash(upload_run.run_dir / 'grt' / 'last.pt') == checkpoint_before

export = client.request('POST', '/exports', {'job_ids': [upload_job['id']]}, key=test_id + '-export')
export = wait_for(lambda: client.request('GET', '/exports/' + export['id']), lambda e: e['status'] != 'preparing')
assert export['status'] == 'ready'
archive = collected / 'results.zip'
archive.write_bytes(transfer('GET', export['download']['url']).content)
from zipfile import ZipFile
with ZipFile(archive) as bundle:
    assert bundle.testzip() is None
    zip_digests = {hashlib.sha256(bundle.read(n)).hexdigest() for n in bundle.namelist() if not n.endswith('/')}
assert {a['sha256'] for a in verified} <= zip_digests
record('download_zip_repeat', job_id=upload_job['id'], verified_files=len(verified))


## 5. 작은 posthoc 실패 → 분석만 재시도 (선택, R07/F09)

의도적으로 잘못된 분석 window를 한 번 전달합니다. 실패 후에도 원본 학습 report가 서버에 남고, 올바른 옵션으로 분석만 재시도한 뒤 성공해야 합니다. 기본값은 켜져 있습니다.


In [ ]:
if run_posthoc:
    analysis_run = make_run('posthoc')
    analysis_run.setup(install=False)
    analysis_run.prepare_data(prepared_input=prepared)
    analysis_run.preflight()
    analysis_run.train()
    checkpoint_before = file_hash(analysis_run.run_dir / 'grt' / 'last.pt')
    try:
        analysis_run.report(plot=False, posthoc=True, posthoc_options={'window_starts': [0, 1]})
    except subprocess.CalledProcessError:
        pass
    else:
        raise AssertionError('잘못된 분석 window가 거부되어야 합니다.')
    items = client.request('GET', '/jobs?campaign=' + quote(analysis_run.run_dir.name))['data']
    assert len(items) == 1
    partial = track(detail(items[0]['id']))
    assert partial['status'] == 'running' and partial['summary']['phase_error'] == 'posthoc_failed'
    assert any(a['name'] == 'experiment_report.json' and a['status'] == 'verified' for a in partial['artifacts'])
    analysis_run.report(plot=False, posthoc=True, posthoc_options=dict(
        batch_size=2, segments=2, warmup_segments=1, horizon=2,
        window_starts=[0, 6, 12], group_repeats=1, lesion_fractions=[0.5]))
    assert detail(partial['id'])['status'] == 'succeeded'
    assert file_hash(analysis_run.run_dir / 'grt' / 'last.pt') == checkpoint_before
    record('posthoc_retry_without_training', job_id=partial['id'])
else:
    checks['posthoc_retry_without_training'] = {'status': 'NOT_RUN'}


## 6. 큐 2개 등록·pause·우선순위·drain·resume (Q04~Q08)

전용 큐에 작은 작업 두 개를 넣습니다. pause 중에는 배정하지 않고, resume 후 높은 우선순위부터 실행합니다. 첫 작업 중 drain해도 현재 작업은 완료하며 다음 작업은 가져오지 않아야 합니다. 마지막에 resume해서 두 번째 작업을 완료합니다.


In [ ]:
queue_bodies = []
for label, priority in [('queue-low', 10), ('queue-high', 20)]:
    source_run = make_run(label, mode='off')
    source_run.prepare_data(prepared_input=prepared)
    body = source_run.server_job('grt')
    body.update(dispatch='queued', priority=priority)
    queue_bodies.append(body)
client.request('POST', '/queues/' + queue['id'] + '/actions', {'action': 'pause'})
submitted = client.request('POST', '/jobs/batch', {'jobs': queue_bodies}, key=test_id + '-batch')['data']
for item in submitted:
    track(item)
low_id, high_id = [j['id'] for j in submitted]
worker_dir = work / 'queue-worker'

def claim_one():
    return WikiTextRun.claim(repo, work_dir=worker_dir, deadline=deadline,
        runtime=runtime, device=device, world_size=1,
        server_url=server_url, secret_source=secret_source, queue=queue_name)

assert claim_one() is None
assert all(detail(j['id'])['status'] == 'queued' for j in submitted)
record('queue_pause_batch', job_ids=[low_id, high_id])


In [ ]:
client.request('POST', '/queues/' + queue['id'] + '/actions', {'action': 'resume'})
first = claim_one()
assert first is not None and detail(high_id)['status'] == 'running'
worker_id = detail(high_id)['attempts'][-1]['worker_id']
client.request('POST', '/workers/' + worker_id + '/actions', {'action': 'drain'})

def execute_claimed(run):
    try:
        run.benchmark_steps = 1
        run.setup(install=False)
        run.prepare_data(prepared_input=prepared)
        run.preflight()
        run.train()
        run.report(plot=False)
    finally:
        run.close()

execute_claimed(first)
assert detail(high_id)['status'] == 'succeeded'
assert claim_one() is None
assert detail(low_id)['status'] == 'queued'
client.request('POST', '/workers/' + worker_id + '/actions', {'action': 'resume'})
second = claim_one()
assert second is not None and detail(low_id)['status'] == 'running'
execute_claimed(second)
assert detail(low_id)['status'] == 'succeeded'
assert claim_one() is None
record('queue_priority_drain_resume', high_job=high_id, low_job=low_id, worker_id=worker_id)


## 7. clone·queued 취소·retry (Q13)

완료된 작업을 복제하고, queued 상태에서 취소·retry한 뒤 실제로 실행합니다. 원래 성공한 job은 바뀌지 않아야 합니다.


In [ ]:
cloned = track(client.request('POST', '/jobs/' + high_id + '/actions',
    {'action': 'clone', 'overrides': {'name': test_id + '-clone'}}, key=test_id + '-clone'))
assert cloned['id'] != high_id and cloned['status'] == 'queued'
client.request('POST', '/jobs/' + cloned['id'] + '/actions', {'action': 'cancel'})
assert detail(cloned['id'])['status'] == 'cancelled'
client.request('POST', '/jobs/' + cloned['id'] + '/actions', {'action': 'retry'})
assert detail(cloned['id'])['status'] == 'queued'
retry_run = claim_one()
assert retry_run is not None
execute_claimed(retry_run)
assert detail(cloned['id'])['status'] == 'succeeded'
assert detail(high_id)['status'] == 'succeeded'
record('clone_cancel_retry', job_id=cloned['id'])


## 8. 실제 학습 취소 (선택, F01)

첫 셀이 긴 목표 step의 **전용 작은 실험**을 백그라운드에서 시작합니다. 바로 다음 셀은 첫 학습 기록을 기다린 뒤 서버 cancel을 보냅니다. GUI에서 직접 취소해도 됩니다. 취소를 확인하는 셀까지 연속 실행하세요. 설정은 기본 켜짐입니다.


In [ ]:
if run_cancel:
    from concurrent.futures import ThreadPoolExecutor
    cancel_run = make_run('cancel', steps=100000, model='rmt')
    cancel_run.setup(install=False)
    cancel_run.prepare_data(prepared_input=prepared)
    cancel_run.preflight()
    items = client.request('GET', '/jobs?campaign=' + quote(cancel_run.run_dir.name))['data']
    assert len(items) == 1
    cancel_id = track(items[0])['id']
    pool = ThreadPoolExecutor(max_workers=1)
    future = pool.submit(cancel_run.train)
    print('학습 시작:', cancel_id, '→ 바로 다음 취소 셀을 실행하세요.')
else:
    checks['running_cancel_checkpoint'] = {'status': 'NOT_RUN'}


In [ ]:
if run_cancel:
    metrics = cancel_run.run_dir / 'rmt' / 'metrics.jsonl'
    try:
        def trained_steps():
            if not metrics.exists():
                return 0
            rows = metrics.read_text().splitlines()
            for row in reversed(rows):
                try:
                    step = json.loads(row).get('step', 0)
                    if step > 0:
                        return step
                except json.JSONDecodeError:
                    pass  # writer가 기록 중인 마지막 줄은 다음 poll에서 확인
            return 0
        wait_for(trained_steps, lambda step: step > 0, timeout=120)
        client.request('POST', '/jobs/' + cancel_id + '/actions', {'action': 'cancel'})
        try:
            future.result(timeout=150)
        except ServerError as error:
            if error.code != 'cancel_requested':
                raise
        else:
            cancel_run.report(plot=False)
        stopped = wait_for(lambda: detail(cancel_id), lambda j: j['status'] not in ('running', 'finalizing'))
        assert stopped['status'] == 'cancelled'
        checkpoint = torch.load(cancel_run.run_dir / 'rmt' / 'last.pt', map_location='cpu', weights_only=False)
        assert 0 < checkpoint['step'] < 100000
        assert any(a['kind'] == 'checkpoint' and a['status'] == 'verified' for a in stopped['artifacts'])
        record('running_cancel_checkpoint', job_id=cancel_id, saved_step=checkpoint['step'])
    finally:
        # 시간 초과 시에도 cancel 요청은 남겨 두며, 끝난 프로세스만 정리합니다.
        client.request('POST', '/jobs/' + cancel_id + '/actions', {'action': 'cancel'}) if detail(cancel_id)['status'] == 'running' else None
        if future.done():
            cancel_run.close()
        pool.shutdown(wait=False)


## 9. API 빠른 오류 검사 (A03~A05, Q02~Q04, F11)

읽기 전용 토큰을 메모리에만 발급해서 권한·쿠키 CSRF·폐기 반영을 검사하고 바로 폐기합니다. 원래 플랫폼 Secret과 비밀번호는 변경하지 않습니다. 잘못된 batch는 전부 rollback되어야 합니다.


In [ ]:
valid = copy.deepcopy(queue_bodies[0])
client.request('POST', '/jobs/validate', valid)
invalid = copy.deepcopy(valid)
invalid['spec']['target_steps'] = 0
try:
    client.request('POST', '/jobs/validate', invalid)
except ServerError as error:
    assert error.status == 422
else:
    raise AssertionError('잘못된 target_steps 허용')
batch_before = client.request('GET', '/jobs?queue=' + quote(queue_name))['data']
try:
    client.request('POST', '/jobs/batch', {'jobs': [valid, invalid]}, key=test_id + '-bad-batch')
except ServerError as error:
    assert error.status == 422
else:
    raise AssertionError('잘못된 batch 허용')
assert {j['id'] for j in client.request('GET', '/jobs?queue=' + quote(queue_name))['data']} == {j['id'] for j in batch_before}

same_key = test_id + '-same-request'
one = track(client.request('POST', '/jobs', valid, key=same_key))
two = client.request('POST', '/jobs', valid, key=same_key)
assert one['id'] == two['id']
changed = copy.deepcopy(valid)
changed['name'] += '-changed'
try:
    client.request('POST', '/jobs', changed, key=same_key)
except ServerError as error:
    assert error.status == 409
else:
    raise AssertionError('같은 key의 다른 본문 허용')
request_record = client.request('GET', '/requests/' + same_key)
client.request('POST', '/jobs/' + one['id'] + '/actions', {'action': 'cancel'})
record('validation_atomic_batch_idempotency', job_id=one['id'])


In [ ]:
issued = client.request('POST', '/tokens', {'name': test_id + '-reader', 'kind': 'read-only'})
reader_id = issued['id']
reader_headers = {'Authorization': 'Bearer ' + issued['token']}
browser = requests.Session()
try:
    read_response = requests.get(server_url + '/v1/jobs/' + upload_job['id'], headers=reader_headers, timeout=20)
    assert read_response.status_code == 200
    denied = requests.post(server_url + '/v1/queues', headers=reader_headers,
                           json={'name': test_id + '-forbidden'}, timeout=20)
    assert denied.status_code == 403
    allowed = requests.post(server_url + '/v1/jobs/validate', headers=reader_headers, json=valid, timeout=20)
    assert allowed.status_code == 200
    login = browser.post(server_url + '/v1/sessions', headers={'Origin': server_url},
                         json={'token': issued['token']}, timeout=20)
    assert login.status_code == 201
    csrf = login.json()['csrf_token']
    export_body = {'job_ids': [upload_job['id']]}
    missing = browser.post(server_url + '/v1/exports', headers={'Origin': server_url}, json=export_body, timeout=20)
    assert missing.status_code == 403
    wrong_origin = browser.post(server_url + '/v1/exports',
        headers={'Origin': 'https://invalid.example', 'X-CSRF-Token': csrf}, json=export_body, timeout=20)
    assert wrong_origin.status_code == 403
    allowed = browser.post(server_url + '/v1/exports',
        headers={'Origin': server_url, 'X-CSRF-Token': csrf, 'Idempotency-Key': test_id + '-reader-export'},
        json=export_body, timeout=20)
    assert allowed.status_code in (200, 202)  # 같은 파일 ZIP이 이미 ready이면 200
    client.request('DELETE', '/tokens/' + reader_id)
    assert requests.get(server_url + '/v1/me', headers=reader_headers, timeout=20).status_code == 401
    assert browser.get(server_url + '/v1/me', timeout=20).status_code == 401
    record('read_only_csrf_revocation')
finally:
    client.request('DELETE', '/tokens/' + reader_id)
    browser.close()
    del issued, reader_headers


## 10. 잘못된 작은 파일은 verified가 되지 않는지 (B02)

별도의 몇 byte짜리 dataset 파일을 예약하고 틀린 SHA256을 전달합니다. 연구 파일을 변조하지 않습니다. 기본 local 저장에서 실패 검증을 확인합니다. S3는 PUT 단계에서 checksum을 거부할 수도 있습니다.


In [ ]:
payload = b'acceptance-wrong-hash'
reservation = client.request('POST', '/artifacts/uploads', dict(name='acceptance-bad-hash.txt',
    kind='dataset', size_bytes=len(payload), sha256='0' * 64, format='txt', format_version='1'))
try:
    put_response = requests.put(reservation['upload']['url'], data=payload,
                               headers=reservation['upload'].get('headers', {}), timeout=30, allow_redirects=False)
except requests.RequestException:
    raise RuntimeError('잘못된 파일 시험의 전송 실패; 서명 URL은 공유하지 마세요.') from None
if put_response.ok:
    client.request('POST', '/artifacts/uploads/' + reservation['id'] + '/complete', {})
    failed = wait_for(lambda: client.request('GET', '/artifacts/uploads/' + reservation['id']), lambda a: a['status'] != 'verifying')
    assert failed['status'] == 'failed'
    assert failed['error']['code'] == 'sha256_mismatch'
    record('bad_hash_rejected', artifact_id=reservation['id'])
elif put_response.status_code == 400 and caps.get('storage') == 's3':
    checks['bad_hash_rejected'] = {'status': 'NEEDS_S3_REVIEW', 'http_status': 400}
    print('S3 PUT이 거부되었습니다. checksum 오류인지 응답을 비공개로 확인하세요.')
else:
    raise AssertionError('예상하지 못한 PUT 상태: ' + str(put_response.status_code))


## 11. 결과 요약 (중간 실패 후에도 실행 가능)

이 파일만 공유하면 됩니다. 비밀값·서명 URL·lease·전체 Config는 넣지 않습니다. 중간에 실패한 셀은 PASS로 기록되지 않습니다. 실제 상태가 running인 job이 남으면 해당 학습 프로세스/런타임을 확인한 뒤 처리하세요. 단순 화면 timeout은 프로세스 종료 증거가 아닙니다.

아직 남은 수동 검사: 브라우저 비밀번호·세션 관리, 다른 플랫폼, NeoX/RMT 정상 완료, 실제 중간 5분 checkpoint, 구세대 차단·토큰 교체·장기 단절, CLI, EC2 재시작·백업 복원. [실행 가이드](https://github.com/YeoJune/GRT_server/blob/main/docs/20261023_aws_test_execution_guide.md)로 이어갑니다.


In [ ]:
expected_checks = ['connection', 'prepared', 'upload', 'download_zip_repeat',
    'posthoc_retry_without_training', 'queue_pause_batch', 'queue_priority_drain_resume',
    'clone_cancel_retry', 'running_cancel_checkpoint', 'validation_atomic_batch_idempotency',
    'read_only_csrf_revocation', 'bad_hash_rejected']
job_states = []
for job_id in tracked_jobs:
    try:
        item = detail(job_id)
        job_states.append(dict(job_id=job_id, status=item['status'],
            attempts=[dict(attempt_id=a['id'], generation=a['generation'], status=a['status']) for a in item['attempts']]))
    except ServerError as error:
        job_states.append(dict(job_id=job_id, query_error=error.code))
summary = dict(test_id=test_id, utc=datetime.now(timezone.utc).isoformat(), server_url=server_url,
    grt_commit=commit, runtime=runtime, device=device, world_size=1, input=data_mode,
    gpu=torch.cuda.get_device_name(0) if device == 'cuda' else None,
    checks={**{name: {'status': 'NOT_RUN_OR_FAILED'} for name in expected_checks}, **checks},
    jobs=job_states, remaining_manual_tests='See AWS execution guide; notebook PASS is not full acceptance')
summary_path = work / 'acceptance_summary.json'
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2))
for name, result in summary['checks'].items():
    print(result['status'], name)
print('공유할 파일:', summary_path)
if runtime == 'colab':
    from google.colab import files
    files.download(str(summary_path))
else:
    from IPython.display import FileLink, display
    display(FileLink(str(summary_path)))
